In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Locate the repo root without importing from src yet.
_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (_current, *_current.parents)
    if (candidate / "AGENTS.md").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection.data_loading import load_split, baseline_mean_metrics


In [3]:
X_train, y_train = load_split("train", processed_dir=PROJECT_ROOT / "data" / "processed")
X_val, y_val = load_split("validation", processed_dir=PROJECT_ROOT / "data" / "processed")

feature_cols = list(X_train.columns)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)


Train: (1895, 25)
Validation: (600, 25)


## 1. Fit once on train, rank all 25 features by impurity importance

In [4]:
RF_PARAMS = dict(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=15,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
)

rf = RandomForestRegressor(**RF_PARAMS)
rf.fit(X_train, y_train)

importance_df = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance": rf.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

importance_df


,feature,importance
0,sma_60,0.146339
1,sma_5,0.106009
2,volatility_20,0.098666
3,sma_20,0.090085
4,atr_14,0.073463
5,volume_sma_20,0.050372
6,macd_signal,0.042527
7,price_to_sma_60,0.041649
8,macd_hist,0.041190
9,rsi_14,0.035122


## 2. Top-K subsets vs. the train-mean baseline

Refit a fresh RF (same hyperparameters) restricted to the top-K
features by importance, evaluate once on validation.

In [5]:
top_k_list = [5, 10, 15, 20]

results = [baseline_mean_metrics(y_train, y_val)]

for k in top_k_list:
    top_features = importance_df["feature"].head(k).tolist()

    model = RandomForestRegressor(**RF_PARAMS)
    model.fit(X_train[top_features], y_train)

    y_pred = model.predict(X_val[top_features])

    mse = mean_squared_error(y_val, y_pred)

    results.append({
        "method": "RandomForest Importance",
        "n_selected_features": k,
        "selected_features": top_features,
        "RMSE": mse ** 0.5,
        "MAE": mean_absolute_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred),
    })

rf_results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
rf_results_df


,method,n_selected_features,selected_features,RMSE,MAE,R2
0,RandomForest Importance,5,"[sma_60, sma_5, volatility_20, sma_20, atr_14]",0.102215,0.075798,0.046587
1,RandomForest Importance,10,"[sma_60, sma_5, volatility_20, sma_20, atr_14,...",0.102833,0.076869,0.035017
2,RandomForest Importance,15,"[sma_60, sma_5, volatility_20, sma_20, atr_14,...",0.103099,0.077286,0.030026
3,RandomForest Importance,20,"[sma_60, sma_5, volatility_20, sma_20, atr_14,...",0.103205,0.077510,0.028028
4,Baseline (predict train mean),0,[],0.105132,0.077442,-0.008612


In [6]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "embedded_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

importance_df.to_csv(OUTPUT_DIR / "random_forest_importance_full.csv", index=False)
rf_results_df.to_csv(OUTPUT_DIR / "random_forest_importance_results.csv", index=False)

print("Saved to:", OUTPUT_DIR)


Saved to: /Users/yangjaehoon/Desktop/StockLens/data/processed/embedded_results
